# Monitors & Reports

This file sets up the model, data, and infrastructure monitors. It also establishes a monitoring dashboard the code for generating reports on SageMaker.

Attribution: The code was made with the assistance of Perplexity.ai accessed in February 2026.

### Set Up Model & Baseline

In [1]:
#Import key libraries

#Sagemaker Imports
import sagemaker
from sagemaker import Session
from sagemaker.model_monitor import (
    DefaultModelMonitor,
    ModelQualityMonitor,
    DatasetFormat,
    CronExpressionGenerator,
    EndpointInput
)
from sagemaker.model import Model
from sagemaker.predictor import Predictor
from sagemaker.serializers import IdentitySerializer
from sagemaker.deserializers import JSONDeserializer
from sagemaker import image_uris, get_execution_role

#Other Imports
import s3fs
import pandas as pd
import boto3
import json
import time
import tarfile
import botocore.exceptions
import io
from io import StringIO
from datetime import datetime, timedelta
import numpy as np
from pathlib import Path

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml


In [2]:
#Set up session
region = "us-east-1"
session = Session()
sm_client = boto3.client("sagemaker", region_name=region)
cw_client = boto3.client("cloudwatch", region_name=region)
s3_client = boto3.client("s3", region_name=region)

role = sagemaker.get_execution_role()
bucket = "sagemaker-us-east-1-418418308994"
prefix = "models/benchmarks"

In [3]:
# Config
bucket = "sagemaker-us-east-1-418418308994"
local_base = Path('/tmp/Models/benchmarks')
s3_base = f"s3://{bucket}/models/benchmarks"

print("Loading XGBoost artifacts...")

# XGBoost paths (prioritize local)
xgb_paths = {
    'local_tar.gz': local_base / 'xgboost/model.tar.gz',
    's3_tar.gz': f"{s3_base}/xgboost/model.tar.gz",
    's3_pkl': f"{s3_base}/xgboost/model.pkl",
    'local_metrics': local_base / 'xgboost/metrics.json',
    'local_model': local_base / 'xgboost/model.pkl'
}

# Auto-select best path
model_data = str(xgb_paths['local_tar.gz']) if xgb_paths['local_tar.gz'].exists() else xgb_paths['s3_tar.gz']
metrics_path = xgb_paths['local_metrics'] if xgb_paths['local_metrics'].exists() else None

print(f"Model data: {model_data}")
print(f"Metrics: {metrics_path}")
print("XGBoost paths ready!")

🔍 Loading XGBoost artifacts...
Model data: s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz
Metrics: None
XGBoost paths ready!


In [4]:
# Get the latest XGBoost container for region
xgboost_container = image_uris.retrieve(
    framework="xgboost",
    region="us-east-1",
    version="1.7-1" 
)

print(f"Using XGBoost container: {xgboost_container}")

# Create model with updated container
new_xgb_endpoint_name = f"xgb-benchmark-endpoint-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

xg_model_new = Model(
    image_uri=xgboost_container,
    model_data="s3://sagemaker-us-east-1-418418308994/models/benchmarks/xgboost/model.tar.gz",
    role=role,
    sagemaker_session=session,
)

print(f"Deploying new endpoint: {new_xgb_endpoint_name}")

# Deploy with data capture
data_capture_prefix = f"{prefix}/data-capture"
data_capture_s3_uri = f"s3://{bucket}/{data_capture_prefix}"

xg_predictor_new = xg_model_new.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=new_xgb_endpoint_name,
    data_capture_config=sagemaker.model_monitor.DataCaptureConfig(
        enable_capture=True,
        sampling_percentage=100,
        destination_s3_uri=data_capture_s3_uri,
        capture_options=["REQUEST", "RESPONSE"],
    ),
)

print(f"Endpoint deployed: {new_xgb_endpoint_name}")

Using XGBoost container: 683313688378.dkr.ecr.us-east-1.amazonaws.com/sagemaker-xgboost:1.7-1
Deploying new endpoint: xgb-benchmark-endpoint-20260208-041551
------!Endpoint deployed: xgb-benchmark-endpoint-20260208-041551


In [5]:
# Wait for endpoint to be in service
print("Waiting for endpoint to be ready...")

sm_client = boto3.client('sagemaker', region_name='us-east-1')

while True:
    response = sm_client.describe_endpoint(EndpointName=new_xgb_endpoint_name)
    status = response['EndpointStatus']
    print(f"  Status: {status}")
    
    if status == 'InService':
        print("Endpoint is ready!")
        break
    elif status == 'Failed':
        print("Endpoint deployment failed!")
        print(f"Failure reason: {response.get('FailureReason', 'Unknown')}")
        break
    
    time.sleep(30)

Waiting for endpoint to be ready...
  Status: InService
✅ Endpoint is ready!


In [6]:
# Create predictor for the new endpoint
xg_predictor_new = Predictor(
    endpoint_name=new_xgb_endpoint_name,
    sagemaker_session=session,
)

# Load test data
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
test_data = baseline_df.drop("target", axis=1).iloc[:5]

print(f"Test data shape: {test_data.shape}")

# Convert to CSV
csv_payload = test_data.to_csv(header=False, index=False)

# Send prediction
response = xg_predictor_new.predict(
    data=csv_payload,
    initial_args={
        'ContentType': 'text/csv',
        'Accept': 'text/csv'
    }
)

print(f"✅ Predictions: {response}")

Test data shape: (5, 20)
✅ Predictions: b'0.009759249165654182,0.010805039666593075,0.48084768652915955,0.03209933266043663,0.03439685329794884,0.4277508556842804,0.004340957384556532\n0.007653615437448025,0.007332483772188425,0.22988218069076538,0.07651730626821518,0.07866891473531723,0.5871987342834473,0.012746804393827915\n0.01511878240853548,0.01036760676652193,0.5404502153396606,0.03245336189866066,0.12612764537334442,0.2628888487815857,0.012593579478561878\n0.015056170523166656,0.004233742132782936,0.17002418637275696,0.01482530776411295,0.09608134627342224,0.6938716769218445,0.005907570943236351\n0.02310902625322342,0.016121942549943924,0.2598908543586731,0.01709647849202156,0.08900441229343414,0.5792668461799622,0.015510435216128826\n'


In [14]:
#Use if endpoint already deployed
xgb_endpoint_name = "xgb-benchmark-endpoint-20260208-041551"

In [8]:
try:
    dq_stats_uri = dq_monitor.latest_baselining_job.baseline_statistics.file_name
    dq_constraints_uri = dq_monitor.latest_baselining_job.suggested_constraints.file_name
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")
except:
    # List S3 output manually if SDK fails
    s3 = boto3.client('s3')
    response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
    for obj in response.get('Contents', []):
        if obj['Key'].endswith('statistics.json'):
            dq_stats_uri = f"s3://{bucket}/{obj['Key']}"
        if obj['Key'].endswith('constraints.json'):
            dq_constraints_uri = f"s3://{bucket}/{obj['Key']}"
    print(f"Stats: {dq_stats_uri}")
    print(f"Constraints: {dq_constraints_uri}")

Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json
Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json


In [9]:
#Find JSON files
s3 = boto3.client('s3')

response = s3.list_objects_v2(Bucket=bucket, Prefix=f"{prefix}/monitoring/baseline/")
files = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.json')]
print("JSON files:")
for f in files:
    print(f"s3://{bucket}/{f}")

JSON files:
s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json
s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json


In [10]:
#Check what constraints look like
key = "models/benchmarks/monitoring/baseline/constraints.json"
obj = s3_client.get_object(Bucket=bucket, Key=key)
constraints = json.load(obj['Body'])

print("Full structure:")
print(json.dumps(constraints, indent=2)[:1000])  # First 1000 chars

constraints_df = pd.json_normalize(constraints['features'])
print("\nAvailable columns:")
print(constraints_df.columns.tolist())
print("\nFirst 5 rows:")
print(constraints_df.head())

Full structure:
{
  "version": 0.0,
  "features": [
    {
      "name": "meanfreq",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "sd",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "median",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "q25",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": true
      }
    },
    {
      "name": "q75",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints": {
        "is_non_negative": false
      }
    },
    {
      "name": "iqr",
      "inferred_type": "Fractional",
      "completeness": 1.0,
      "num_constraints":

In [11]:
#Check what statistics look like
key = "models/benchmarks/monitoring/baseline/statistics.json"
obj = s3_client.get_object(Bucket=bucket, Key=key)
statistics = json.load(obj['Body'])

stats_df = pd.json_normalize(statistics['features'])
print("Audio Feature Statistics (Top 10):")
print(stats_df[['name', 
                'numerical_statistics.mean', 
                'numerical_statistics.std_dev', 
                'numerical_statistics.min', 
                'numerical_statistics.max',
                'numerical_statistics.completeness']].head(10))

Audio Feature Statistics (Top 10):
       name  numerical_statistics.mean  numerical_statistics.std_dev  \
0  meanfreq                2348.249412                   1468.419325   
1        sd                2447.622411                   1105.571653   
2    median                4593.042497                   2696.537284   
3       q25                   0.090211                      0.045230   
4       q75                -404.312476                    106.927208   
5       iqr                  87.612007                     25.563514   
6      skew                  19.789872                     22.195497   
7      kurt                  17.906020                     11.754694   
8    sp_ent                   3.565427                     10.358779   
9       sfm                   1.530392                      7.320638   

   numerical_statistics.min  numerical_statistics.max  \
0                  0.000000               7655.335725   
1                  0.000000               6324.351767   
2

## Data Quality Monitoring

In [12]:
#Ensure monitor isn't already in place
try:
    dq_monitor.delete_monitoring_schedule(schedule_name_xgb_dq)
    print(f"Deleted existing: {schedule_name_xgb_dq}")
except:
    print("No existing schedule")

No existing schedule


### Set Up Data Quality Monitor

In [19]:
role = get_execution_role()

#Create the monitor
dq_monitor = DefaultModelMonitor(
    role=role,
    instance_count=1,
    instance_type='ml.m5.xlarge',
    volume_size_in_gb=20,
    max_runtime_in_seconds=3600,
)

#Create schedule (using daily instead of hourly to save costs!)
schedule_name_xgb_dq = "xgb-data-quality-schedule"

#Hourly Monitoring Schedule
dq_monitor.create_monitoring_schedule(
    monitor_schedule_name=schedule_name_xgb_dq,
    endpoint_input=xgb_endpoint_name,  # Make sure this variable is defined
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/data-quality/xgb",
    statistics="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/statistics.json",
    constraints="s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/baseline/constraints.json",
    schedule_cron_expression=CronExpressionGenerator.hourly(), #Hourly
    enable_cloudwatch_metrics=True,
)

print(f"Daily schedule live: {schedule_name_xgb_dq}")

Daily schedule live: xgb-data-quality-schedule


In [20]:
#Check status of endpoint
sm_client = boto3.client('sagemaker')
response = sm_client.describe_endpoint(EndpointName=xgb_endpoint_name)
print("Status:", response['EndpointStatus'])
print("Last heartbeat:", response.get('LastHeartbeatTimestamp', 'N/A'))

Status: InService
Last heartbeat: N/A


In [ ]:
#Check that DataCapture is writing to S3

#Create predictor
xg_predictor = Predictor(
    endpoint_name=new_xgb_endpoint_name,
    sagemaker_session=session,
)

# Load test data (scaled)
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
test_data = baseline_df.drop("target", axis=1).iloc[:100]  # Send 100 samples

print(f"Sending {len(test_data)} predictions to generate monitoring data...")

# Send predictions
csv_payload = test_data.to_csv(header=False, index=False)
response = xg_predictor.predict(
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

print("✅✅✅ Predictions sent!")
print("Waiting for data capture to write to S3 (this takes a few minutes)...")
time.sleep(300)  #Wait a few minutes for data capture

## Model Quality Moniter

In [23]:
#Get predictions (batch all rows)
X_baseline = baseline_df.drop('target', axis=1)
csv_payload = X_baseline.to_csv(header=False, index=False)

#Get predictions from endpoint
response = xg_predictor_new.predict(  
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

# Parse CSV predictions (format: prob1,prob2,...,prob7\n per row)
predictions_text = response.decode('utf-8').strip().split('\n')
prob_matrix = np.array([[float(x) for x in line.split(',') if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)

# Create MQ baseline dataframe
mq_baseline_df = pd.DataFrame({
    'prediction': pred_labels,
    'ground_truth_label': baseline_df['target'].values
})

# Save using s3_client instead
mq_baseline_key = "models/benchmarks/mq_baseline.csv"
csv_buffer = StringIO()
mq_baseline_df.to_csv(csv_buffer, index=False, header=True)
s3_client.put_object(
    Bucket=bucket,
    Key=mq_baseline_key,
    Body=csv_buffer.getvalue()
)
mq_baseline_uri = f"s3://{bucket}/{mq_baseline_key}"
print(f"MQ baseline: {mq_baseline_uri} ({len(mq_baseline_df)} rows)")

MQ baseline: s3://sagemaker-us-east-1-418418308994/models/benchmarks/mq_baseline.csv (10242 rows)


In [24]:
# Create Model Quality Monitor
mq_monitor = ModelQualityMonitor(
    role=role, 
    instance_count=1, 
    instance_type="ml.m5.xlarge",
    volume_size_in_gb=20, 
    max_runtime_in_seconds=3600, 
    sagemaker_session=session,
)

# Define problem type and constraints
model_quality_baseline_uri = f"s3://{bucket}/{prefix}/monitoring/mq-baseline"

In [25]:
#Suggest baseline for model quality
mq_monitor.suggest_baseline(
    baseline_dataset=mq_baseline_uri,
    dataset_format=DatasetFormat.csv(header=True),
    output_s3_uri=model_quality_baseline_uri,
    problem_type='MulticlassClassification',
    inference_attribute='prediction',  
    ground_truth_attribute='ground_truth_label',  
    wait=True,
    logs=False
)

print("\n✅✅ \nModel Quality baseline created! \n✅✅")

INFO:sagemaker:Creating processing-job with name baseline-suggestion-job-2026-02-08-04-29-33-007


...........................................................!Model Quality baseline created!


In [28]:
# Get the baselining job description
job_description = mq_monitor.latest_baselining_job.describe()

# Extract the S3 URIs
mq_stats_uri = job_description['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri'] + '/statistics.json'
mq_constraints_uri = job_description['ProcessingOutputConfig']['Outputs'][0]['S3Output']['S3Uri'] + '/constraints.json'

print(f"Stats: {mq_stats_uri}")
print(f"Constraints: {mq_constraints_uri}")

Stats: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline/statistics.json
Constraints: s3://sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/mq-baseline/constraints.json


In [36]:
# Fake production ground truth
ground_truth_key = f"ground-truth-{datetime.now().strftime('%Y%m%d-%H')}.jsonl"
fake_gt = []
for i, gt_label in enumerate(mq_baseline_df['ground_truth_label']):
    fake_gt.append(json.dumps({
        "groundTruthData": {
            "data": str(int(gt_label)),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": f"lab-{i}",
            "inferenceTime": (datetime.now() - timedelta(hours=1)).isoformat()
        },
        "eventVersion": "0"
    }))

# Upload ground truth
s3_resource = boto3.resource('s3')
s3_resource.Object(bucket, f"{prefix}/ground-truth/{ground_truth_key}").put(Body='\n'.join(fake_gt))
print(f"Ground truth uploaded: s3://{bucket}/{prefix}/ground-truth/{ground_truth_key}")

# Create Hourly monitoring schedule with inference_attribute
mq_monitor.create_monitoring_schedule(
    monitor_schedule_name="xgb-model-quality-schedule",
    endpoint_input=EndpointInput(
        endpoint_name=new_xgb_endpoint_name,
        destination="/opt/ml/processing/input/endpoint",
        inference_attribute="0",  # For CSV output, use "0" for the prediction column
    ),
    ground_truth_input=f"s3://{bucket}/{prefix}/ground-truth/",
    output_s3_uri=f"s3://{bucket}/{prefix}/monitoring/model-quality/xgb",
    schedule_cron_expression=CronExpressionGenerator.hourly(), #Hourly
    enable_cloudwatch_metrics=True,
    problem_type='MulticlassClassification',
)
print("\n \nFull model quality monitoring active! ✅✅✅✅")

Ground truth uploaded: s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/ground-truth-20260208-04.jsonl


INFO:sagemaker.model_monitor.model_monitoring:Creating Monitoring Schedule with name: xgb-model-quality-schedule


✅ Full model quality monitoring active!


## Model Bias Monitor

In [42]:
print("Skipping bias monitoring - no demographic features in dataset")

Skipping bias monitoring - no demographic features in dataset


## Infrastructure Monitors

In [37]:
# Create CloudWatch client
cw_client = boto3.client('cloudwatch')

# Infrastructure monitoring - CloudWatch alarms for XGBoost endpoint
endpoint_metric_dimensions = [
    {"Name": "EndpointName", "Value": new_xgb_endpoint_name},  # Using your current endpoint variable
]

# Alarm 1: High latency
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-High-Latency",
    AlarmDescription="Model latency for XGBoost endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="ModelLatency",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Average",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=10000.0,  # 10 seconds in milliseconds
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,  # Set to True if you want to enable SNS notifications
)
print("CloudWatch alarm created for XGBoost endpoint latency")

# Alarm 2: Invocation errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-Invocation-Errors",
    AlarmDescription="Invocation errors for XGBoost endpoint",
    Namespace="AWS/SageMaker",
    MetricName="ModelInvocationErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,  # 5 minutes
    EvaluationPeriods=1,
    Threshold=5.0,  # Alert if more than 5 errors in 5 minutes
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,  # Set to True if you want to enable SNS notifications
)
print("CloudWatch alarm created for XGBoost endpoint invocation errors")

# Alarm 3: 5XX server errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-5XX-Errors-High",
    AlarmDescription="5XX error rate for XGBoost endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="Model5XXErrors",  # Changed from ModelLatency
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=5.0,  # Alert if more than 5 5XX errors
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
print("CloudWatch alarm created for XGBoost endpoint 5XX errors")

# Alarm 4: 4XX client errors
cw_client.put_metric_alarm(
    AlarmName="XGB-Endpoint-4XX-Errors-High",
    AlarmDescription="4XX error rate for XGBoost endpoint above threshold",
    Namespace="AWS/SageMaker",
    MetricName="Model4XXErrors",
    Dimensions=endpoint_metric_dimensions,
    Statistic="Sum",
    Period=300,  # 5 minutes
    EvaluationPeriods=2,
    Threshold=10.0,  # Alert if more than 10 4XX errors
    ComparisonOperator="GreaterThanThreshold",
    TreatMissingData="notBreaching",
    ActionsEnabled=False,
)
print("CloudWatch alarm created for XGBoost endpoint 4XX errors")

CloudWatch alarm created for XGBoost endpoint latency
CloudWatch alarm created for XGBoost endpoint invocation errors
CloudWatch alarm created for XGBoost endpoint 5XX errors
CloudWatch alarm created for XGBoost endpoint 4XX errors


In [38]:
#Set each alarm state manually as a test
cw_client = boto3.client('cloudwatch')

# Test each alarm by manually setting it to ALARM state
alarms_to_test = [
    "XGB-Endpoint-High-Latency",
    "XGB-Endpoint-Invocation-Errors",
    "XGB-Endpoint-5XX-Errors-High",
    "XGB-Endpoint-4XX-Errors-High"
]

print("Testing alarms by triggering them manually...\n")

for alarm_name in alarms_to_test:
    # Set alarm to ALARM state
    cw_client.set_alarm_state(
        AlarmName=alarm_name,
        StateValue='ALARM',
        StateReason='Testing alarm - manually triggered from notebook',
        StateReasonData=f'{{"test": "true", "timestamp": "{datetime.now().isoformat()}"}}'
    )
    print(f"Triggered: {alarm_name} -> ALARM state")
    
    # Check the alarm state
    response = cw_client.describe_alarms(AlarmNames=[alarm_name])
    alarm = response['MetricAlarms'][0]
    print(f"   State: {alarm['StateValue']}")
    print(f"   Reason: {alarm['StateReason']}\n")

print("\n" + "="*60)
print("All alarms triggered! Check CloudWatch console to verify.")
print("="*60)

# Optional: Reset all alarms back to OK after testing
print("\nResetting alarms to OK state...\n")
for alarm_name in alarms_to_test:
    cw_client.set_alarm_state(
        AlarmName=alarm_name,
        StateValue='OK',
        StateReason='Test completed - resetting alarm',
    )
    print(f"Reset: {alarm_name} -> OK state")

Testing alarms by triggering them manually...

Triggered: XGB-Endpoint-High-Latency -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook

Triggered: XGB-Endpoint-Invocation-Errors -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook

Triggered: XGB-Endpoint-5XX-Errors-High -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook

Triggered: XGB-Endpoint-4XX-Errors-High -> ALARM state
   State: ALARM
   Reason: Testing alarm - manually triggered from notebook


All alarms triggered! Check CloudWatch console to verify.

Resetting alarms to OK state...

Reset: XGB-Endpoint-High-Latency -> OK state
Reset: XGB-Endpoint-Invocation-Errors -> OK state
Reset: XGB-Endpoint-5XX-Errors-High -> OK state
Reset: XGB-Endpoint-4XX-Errors-High -> OK state


## CloudWatch Monitoring Dashboard

In [39]:
cw_client = boto3.client('cloudwatch')

dashboard_name = "SageMaker-ML-Benchmarks"
dashboard_body = {
    "widgets": [
        {
            "type": "metric",
            "x": 0,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Endpoint – Invocations & Latency",
                "metrics": [
                    ["AWS/SageMaker", "Invocations", "EndpointName", new_xgb_endpoint_name],
                    [".", "ModelLatency", ".", "."],
                ],
                "stacked": False,
                "stat": "Average",
                "period": 60,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 0,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Endpoint – Errors",
                "metrics": [
                    ["AWS/SageMaker", "ModelInvocationErrors", "EndpointName", new_xgb_endpoint_name],
                    [".", "Invocation4XXErrors", ".", "."],
                    [".", "Invocation5XXErrors", ".", "."],
                ],
                "stacked": False,
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Data Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "DataQualityViolation",
                        "MonitoringSchedule",
                        "xgb-data-quality-schedule",
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 12,
            "y": 6,
            "width": 12,
            "height": 6,
            "properties": {
                "title": "XGBoost Model Quality Violations",
                "metrics": [
                    [
                        "AWS/SageMaker",
                        "ModelQualityViolation",
                        "MonitoringSchedule",
                        "xgb-model-quality-schedule",
                    ],
                ],
                "stat": "Sum",
                "period": 300,
                "region": region,
            },
        },
        {
            "type": "metric",
            "x": 0,
            "y": 12,
            "width": 24,
            "height": 6,
            "properties": {
                "title": "CloudWatch Alarms Status",
                "metrics": [
                    ["AWS/CloudWatch", "AlarmState", "AlarmName", "XGB-Endpoint-High-Latency"],
                    ["...", "XGB-Endpoint-Invocation-Errors"],
                    ["...", "XGB-Endpoint-5XX-Errors-High"],
                    ["...", "XGB-Endpoint-4XX-Errors-High"],
                ],
                "stat": "Maximum",
                "period": 300,
                "region": region,
            },
        },
    ]
}

cw_client.put_dashboard(
    DashboardName=dashboard_name,
    DashboardBody=json.dumps(dashboard_body),
)

print(f"CloudWatch Dashboard created: {dashboard_name}")
print(f"View at: https://console.aws.amazon.com/cloudwatch/home?region={region}#dashboards:name={dashboard_name}")

✅ CloudWatch Dashboard created: SageMaker-ML-Benchmarks
View at: https://console.aws.amazon.com/cloudwatch/home?region=us-east-1#dashboards:name=SageMaker-ML-Benchmarks


## Generate Model Traffic

In [44]:
print("Sending test requests to generate metrics...")

# Send 20 requests to generate invocation and latency data
for i in range(20):
    try:
        response = xg_predictor_new.predict(
            data=csv_payload,
            initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
        )
        print(f"Request {i+1}/20 completed")
        time.sleep(2)  # Small delay between requests
    except Exception as e:
        print(f"Request {i+1} failed: {e}")

print("\nTraffic generated! Wait 2-3 minutes then refresh the dashboard.")

Sending test requests to generate metrics...
Request 1/20 completed
Request 2/20 completed
Request 3/20 completed
Request 4/20 completed
Request 5/20 completed
Request 6/20 completed
Request 7/20 completed
Request 8/20 completed
Request 9/20 completed
Request 10/20 completed
Request 11/20 completed
Request 12/20 completed
Request 13/20 completed
Request 14/20 completed
Request 15/20 completed
Request 16/20 completed
Request 17/20 completed
Request 18/20 completed
Request 19/20 completed
Request 20/20 completed

Traffic generated! Wait 2-3 minutes then refresh the dashboard.


In [45]:
print("Generating data quality violations...")

# Load your baseline to see normal ranges
baseline_df = pd.read_csv("s3://sagemaker-us-east-1-418418308994/models/benchmarks/baseline_normalized.csv")
X_baseline = baseline_df.drop('target', axis=1)

# Create anomalous data - extreme values outside normal ranges
anomalous_data = X_baseline.copy()[:10]

# Introduce violations by setting extreme values
for col in anomalous_data.columns[:5]:  # Corrupt first 5 features
    anomalous_data[col] = anomalous_data[col].max() * 100  # 100x the max value
    
print(f"Anomalous data shape: {anomalous_data.shape}")
print(f"Sample extreme values:\n{anomalous_data.iloc[0, :5]}")

# Send anomalous data to endpoint
csv_payload = anomalous_data.to_csv(header=False, index=False)

for i in range(5):
    response = xg_predictor_new.predict(
        data=csv_payload,
        initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
    )
    print(f"Sent anomalous request {i+1}/5")
    time.sleep(1)

print("\n✅ Anomalous data sent! This should trigger data quality violations on next monitoring run.")

Generating data quality violations...
Anomalous data shape: (10, 20)
Sample extreme values:
meanfreq    450709.170872
sd          419013.377644
median      854181.780134
q25             15.419224
q75         -23044.186000
Name: 0, dtype: float64
Sent anomalous request 1/5
Sent anomalous request 2/5
Sent anomalous request 3/5
Sent anomalous request 4/5
Sent anomalous request 5/5

✅ Anomalous data sent! This should trigger data quality violations on next monitoring run.


In [46]:
print("Generating model quality violations...")

# Create fake ground truth with WRONG labels to simulate poor accuracy
s3_resource = boto3.resource('s3')

# Get some predictions
X_test = baseline_df.drop('target', axis=1).iloc[:100]
csv_payload = X_test.to_csv(header=False, index=False)

response = xg_predictor_new.predict(
    data=csv_payload,
    initial_args={'ContentType': 'text/csv', 'Accept': 'text/csv'}
)

# Parse predictions
predictions_text = response.decode('utf-8').strip().split('\n')
prob_matrix = np.array([[float(x) for x in line.split(',') if x] for line in predictions_text])
pred_labels = np.argmax(prob_matrix, axis=1)

# Create INCORRECT ground truth (flip predictions to opposite class)
incorrect_ground_truth = []
for i, pred in enumerate(pred_labels):
    # Assign wrong label - pick a different class than predicted
    wrong_label = (pred + 3) % 7  # Shift by 3 classes to ensure it's wrong
    
    incorrect_ground_truth.append(json.dumps({
        "groundTruthData": {
            "data": str(int(wrong_label)),
            "encoding": "CSV"
        },
        "eventMetadata": {
            "eventId": f"violation-test-{i}",
            "inferenceTime": (datetime.now() - timedelta(hours=1)).isoformat()
        },
        "eventVersion": "0"
    }))

# Upload incorrect ground truth
ground_truth_key = f"ground-truth-violations-{datetime.now().strftime('%Y%m%d-%H%M')}.jsonl"
s3_resource.Object(bucket, f"{prefix}/ground-truth/{ground_truth_key}").put(
    Body='\n'.join(incorrect_ground_truth)
)

print(f"✅\n✅\n✅\n✅ Uploaded incorrect ground truth: s3://{bucket}/{prefix}/ground-truth/{ground_truth_key}")
print("This should trigger model quality violations on next monitoring run (shows 0% accuracy).")

Generating model quality violations...
✅ Uploaded incorrect ground truth: s3://sagemaker-us-east-1-418418308994/models/benchmarks/ground-truth/ground-truth-violations-20260208-0455.jsonl
This should trigger model quality violations on next monitoring run (shows 0% accuracy).


In [47]:
# Manually trigger alarms to show in dashboard
alarms_to_test = [
    "XGB-Endpoint-High-Latency",
    "XGB-Endpoint-Invocation-Errors",
    "XGB-Endpoint-5XX-Errors-High",
    "XGB-Endpoint-4XX-Errors-High"
]

for alarm_name in alarms_to_test:
    cw_client.set_alarm_state(
        AlarmName=alarm_name,
        StateValue='ALARM',
        StateReason='Dashboard test - manually triggered',
    )
    print(f"Triggered: {alarm_name}")

print("\nRefresh dashboard to see alarm states!")

Triggered: XGB-Endpoint-High-Latency
Triggered: XGB-Endpoint-Invocation-Errors
Triggered: XGB-Endpoint-5XX-Errors-High
Triggered: XGB-Endpoint-4XX-Errors-High

Refresh dashboard to see alarm states!


### Generate Model & Data Reports in SageMaker

In [48]:
import s3fs
import json

fs = s3fs.S3FileSystem()

print("="*70)
print("DATA QUALITY MONITORING REPORTS")
print("="*70)

# Get data quality monitoring outputs
latest_dq_output_prefix = f"{bucket}/{prefix}/monitoring/data-quality/xgb"
try:
    dq_reports = fs.ls(latest_dq_output_prefix)
    print(f"\nFound {len(dq_reports)} data quality report files:")
    for report in dq_reports[:10]:  # Show first 10
        print(f"  - {report}")
    
    # Load latest constraint violations
    violations_path = [p for p in dq_reports if p.endswith("constraint_violations.json")]
    if violations_path:
        violations_path = violations_path[-1]
        print(f"\n📊 Latest DQ Violations Report: {violations_path}")
        
        with fs.open(violations_path, "r") as f:
            dq_violations = json.load(f)
        
        print("\nData Quality Violations:")
        print(json.dumps(dq_violations, indent=2))
    else:
        print("\n⚠️ No constraint violations files found yet.")
        print("   The monitoring schedule may not have run yet (it runs daily).")
    
    # Load statistics
    statistics_path = [p for p in dq_reports if p.endswith("statistics.json")]
    if statistics_path:
        statistics_path = statistics_path[-1]
        print(f"\n📈 Latest DQ Statistics: {statistics_path}")
        
        with fs.open(statistics_path, "r") as f:
            dq_statistics = json.load(f)
        
        print("\nData Quality Statistics Summary:")
        if 'features' in dq_statistics:
            print(f"  Features monitored: {len(dq_statistics['features'])}")
            print(f"  Sample feature stats: {list(dq_statistics['features'].keys())[:5]}")
    
except Exception as e:
    print(f"\n⚠️ Error loading data quality reports: {e}")
    print("   The monitoring schedule may not have executed yet.")

print("\n" + "="*70)
print("MODEL QUALITY MONITORING REPORTS")
print("="*70)

# Get model quality monitoring outputs
latest_mq_output_prefix = f"{bucket}/{prefix}/monitoring/model-quality/xgb"
try:
    mq_reports = fs.ls(latest_mq_output_prefix)
    print(f"\nFound {len(mq_reports)} model quality report files:")
    for report in mq_reports[:10]:  # Show first 10
        print(f"  - {report}")
    
    # Load latest constraint violations
    violations_path = [p for p in mq_reports if p.endswith("constraint_violations.json")]
    if violations_path:
        violations_path = violations_path[-1]
        print(f"\n📊 Latest MQ Violations Report: {violations_path}")
        
        with fs.open(violations_path, "r") as f:
            mq_violations = json.load(f)
        
        print("\nModel Quality Violations:")
        print(json.dumps(mq_violations, indent=2))
        
        # Check for specific metrics
        if 'violations' in mq_violations:
            print(f"\n  Total violations: {len(mq_violations['violations'])}")
            for violation in mq_violations['violations'][:5]:  # Show first 5
                print(f"    - {violation}")
    else:
        print("\n⚠️ No constraint violations files found yet.")
        print("   The monitoring schedule may not have run yet (it runs daily).")
    
    # Load statistics
    statistics_path = [p for p in mq_reports if p.endswith("statistics.json")]
    if statistics_path:
        statistics_path = statistics_path[-1]
        print(f"\n📈 Latest MQ Statistics: {statistics_path}")
        
        with fs.open(statistics_path, "r") as f:
            mq_statistics = json.load(f)
        
        print("\nModel Quality Metrics:")
        if 'multiclass_classification_metrics' in mq_statistics:
            metrics = mq_statistics['multiclass_classification_metrics']
            print(f"  Accuracy: {metrics.get('accuracy', {}).get('value', 'N/A')}")
            print(f"  Precision: {metrics.get('precision', {}).get('value', 'N/A')}")
            print(f"  Recall: {metrics.get('recall', {}).get('value', 'N/A')}")
        else:
            print(json.dumps(mq_statistics, indent=2))
    
except Exception as e:
    print(f"\n⚠️ Error loading model quality reports: {e}")
    print("   The monitoring schedule may not have executed yet.")

print("\n" + "="*70)
print("MONITORING EXECUTION STATUS")
print("="*70)

# Check when monitoring last ran
import boto3
sagemaker = boto3.client('sagemaker')

try:
    # Check data quality schedule
    dq_schedule = sagemaker.describe_monitoring_schedule(
        MonitoringScheduleName='xgb-data-quality-schedule'
    )
    print(f"\nData Quality Schedule:")
    print(f"  Status: {dq_schedule['MonitoringScheduleStatus']}")
    print(f"  Last execution: {dq_schedule.get('LastMonitoringExecutionSummary', {}).get('ScheduledTime', 'Not yet run')}")
    
    # Check model quality schedule
    mq_schedule = sagemaker.describe_monitoring_schedule(
        MonitoringScheduleName='xgb-model-quality-schedule'
    )
    print(f"\nModel Quality Schedule:")
    print(f"  Status: {mq_schedule['MonitoringScheduleStatus']}")
    print(f"  Last execution: {mq_schedule.get('LastMonitoringExecutionSummary', {}).get('ScheduledTime', 'Not yet run')}")
    
except Exception as e:
    print(f"Error checking schedule status: {e}")

print("\n" + "="*70)
print("NOTE: If no reports are found, the monitoring schedules haven't run yet.")
print("They are scheduled to run daily. You can wait or manually trigger them.")
print("="*70)

DATA QUALITY MONITORING REPORTS

⚠️ Error loading data quality reports: sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/data-quality/xgb
   The monitoring schedule may not have executed yet.

MODEL QUALITY MONITORING REPORTS

⚠️ Error loading model quality reports: sagemaker-us-east-1-418418308994/models/benchmarks/monitoring/model-quality/xgb
   The monitoring schedule may not have executed yet.

MONITORING EXECUTION STATUS

Data Quality Schedule:
  Status: Scheduled
  Last execution: 2026-02-07 04:00:00+00:00

Model Quality Schedule:
  Status: Scheduled
  Last execution: Not yet run

NOTE: If no reports are found, the monitoring schedules haven't run yet.
They are scheduled to run daily. You can wait or manually trigger them.


# Clean Up

In [ ]:
#Run all safety
"""

In [ ]:
sagemaker = boto3.client('sagemaker')
cw_client = boto3.client('cloudwatch')

print("="*70)
print("CLEANUP SCRIPT - Removing All SageMaker Resources")
print("="*70)

# 1. Delete monitoring schedules
print("\n1. Deleting Monitoring Schedules...")
monitoring_schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
]

for schedule_name in monitoring_schedules:
    try:
        sagemaker.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print(f"  ✅ Deleted: {schedule_name}")
    except sagemaker.exceptions.ResourceNotFound:
        print(f"  ⚠️  Not found: {schedule_name}")
    except Exception as e:
        print(f"  ❌ Error deleting {schedule_name}: {e}")

# 2. Delete endpoint
print("\n2. Deleting Endpoint...")
try:
    sagemaker.delete_endpoint(EndpointName=new_xgb_endpoint_name)
    print(f"  ✅ Deleted endpoint: {new_xgb_endpoint_name}")
except Exception as e:
    print(f"  ❌ Error deleting endpoint: {e}")

# 3. Delete endpoint configuration (optional but recommended)
print("\n3. Deleting Endpoint Configuration...")
try:
    # Get the endpoint config name (usually similar to endpoint name)
    endpoint_config_name = new_xgb_endpoint_name.replace('-endpoint-', '-endpoint-config-')
    sagemaker.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"  ✅ Deleted endpoint config: {endpoint_config_name}")
except Exception as e:
    print(f"  ⚠️  Endpoint config deletion: {e}")

# 4. Delete CloudWatch alarms
print("\n4. Deleting CloudWatch Alarms...")
alarms_to_delete = [
    "XGB-Endpoint-High-Latency",
    "XGB-Endpoint-Invocation-Errors",
    "XGB-Endpoint-5XX-Errors-High",
    "XGB-Endpoint-4XX-Errors-High"
]

try:
    cw_client.delete_alarms(AlarmNames=alarms_to_delete)
    for alarm in alarms_to_delete:
        print(f"  ✅ Deleted alarm: {alarm}")
except Exception as e:
    print(f"  ❌ Error deleting alarms: {e}")

# 5. Delete CloudWatch Dashboard (optional)
print("\n5. Deleting CloudWatch Dashboard...")
try:
    cw_client.delete_dashboards(DashboardNames=["SageMaker-ML-Benchmarks"])
    print(f"  ✅ Deleted dashboard: SageMaker-ML-Benchmarks")
except Exception as e:
    print(f"  ⚠️  Dashboard deletion: {e}")

# 6. Verify cleanup
print("\n" + "="*70)
print("VERIFICATION - Checking for remaining resources")
print("="*70)

# Check endpoints
print("\nRemaining endpoints:")
response = sagemaker.list_endpoints()
if response['Endpoints']:
    for endpoint in response['Endpoints']:
        print(f"  - {endpoint['EndpointName']} (Status: {endpoint['EndpointStatus']})")
else:
    print("  ✅ No endpoints found")

# Check monitoring schedules
print("\nRemaining monitoring schedules:")
response = sagemaker.list_monitoring_schedules()
if response['MonitoringScheduleSummaries']:
    for schedule in response['MonitoringScheduleSummaries']:
        print(f"  - {schedule['MonitoringScheduleName']}")
else:
    print("  ✅ No monitoring schedules found")

# Check CloudWatch alarms
print("\nRemaining CloudWatch alarms (XGB-related):")
try:
    response = cw_client.describe_alarms(AlarmNamePrefix="XGB-")
    if response['MetricAlarms']:
        for alarm in response['MetricAlarms']:
            print(f"  - {alarm['AlarmName']}")
    else:
        print("  ✅ No XGB-related alarms found")
except Exception as e:
    print(f"  ⚠️  Error checking alarms: {e}")

print("\n" + "="*70)
print("CLEANUP COMPLETE!")
print("="*70)
print("\nNote: S3 data (models, baselines, monitoring outputs) is NOT deleted.")
print("If you want to delete S3 data, you'll need to manually clean up:")
print(f"  - s3://{bucket}/{prefix}/")
print("\nCosts should now drop to near-zero (only S3 storage costs).")
print("="*70)

In [50]:
print("="*70)
print("RETRY CLEANUP - Deleting Remaining Resources")
print("="*70)

# Wait for monitoring schedule deletions to propagate
print("\nWaiting 30 seconds for monitoring schedule deletions to propagate...")
time.sleep(30)

# 1. Delete monitoring schedules again (in case they weren't fully deleted)
print("\n1. Ensuring Monitoring Schedules are deleted...")
monitoring_schedules = [
    "xgb-data-quality-schedule",
    "xgb-model-quality-schedule",
]

for schedule_name in monitoring_schedules:
    try:
        sagemaker.delete_monitoring_schedule(MonitoringScheduleName=schedule_name)
        print(f"  ✅ Deleted: {schedule_name}")
    except sagemaker.exceptions.ResourceNotFound:
        print(f"  ✅ Already deleted: {schedule_name}")
    except Exception as e:
        print(f"  ❌ Error: {e}")

# Wait again
print("\nWaiting another 30 seconds...")
time.sleep(30)

# 2. Try deleting endpoint again
print("\n2. Deleting Endpoint...")
endpoint_name = "xgb-benchmark-endpoint-20260208-041551"
try:
    sagemaker.delete_endpoint(EndpointName=endpoint_name)
    print(f"  ✅ Deleted endpoint: {endpoint_name}")
except Exception as e:
    print(f"  ❌ Error deleting endpoint: {e}")

# 3. Delete endpoint configuration with correct name
print("\n3. Deleting Endpoint Configuration...")
endpoint_config_name = "xgb-benchmark-endpoint-config-20260208-041551"  
try:
    sagemaker.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print(f"  ✅ Deleted endpoint config: {endpoint_config_name}")
except sagemaker.exceptions.ResourceNotFound:
    print(f"  ✅ Endpoint config already deleted or doesn't exist")
except Exception as e:
    print(f"  ❌ Error: {e}")

# 4. Final verification
print("\n" + "="*70)
print("FINAL VERIFICATION")
print("="*70)

print("\nRemaining endpoints:")
response = sagemaker.list_endpoints()
if response['Endpoints']:
    for endpoint in response['Endpoints']:
        print(f"  ⚠️  {endpoint['EndpointName']} (Status: {endpoint['EndpointStatus']})")
else:
    print("  ✅ No endpoints found")

print("\nRemaining monitoring schedules:")
response = sagemaker.list_monitoring_schedules()
if response['MonitoringScheduleSummaries']:
    for schedule in response['MonitoringScheduleSummaries']:
        print(f"  ⚠️  {schedule['MonitoringScheduleName']}")
else:
    print("  ✅ No monitoring schedules found")

print("\n" + "="*70)
print("✅ CLEANUP COMPLETE!")
print("="*70)

RETRY CLEANUP - Deleting Remaining Resources

Waiting 30 seconds for monitoring schedule deletions to propagate...

1. Ensuring Monitoring Schedules are deleted...
  ✅ Already deleted: xgb-data-quality-schedule
  ✅ Already deleted: xgb-model-quality-schedule

Waiting another 30 seconds...

2. Deleting Endpoint...
  ✅ Deleted endpoint: xgb-benchmark-endpoint-20260208-041551

3. Deleting Endpoint Configuration...
  ❌ Error: An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not find endpoint configuration "xgb-benchmark-endpoint-config-20260208-041551".

FINAL VERIFICATION

Remaining endpoints:
  ⚠️  xgb-benchmark-endpoint-20260208-041551 (Status: Deleting)

Remaining monitoring schedules:
  ✅ No monitoring schedules found

✅ CLEANUP COMPLETE!
